# Identification in Structural VARs: Cholesky vs. Sign Restrictions

**A Pedagogical Case Study Using U.S. Monetary Policy Data**

A reduced-form VAR captures correlations between macroeconomic variables, but
correlations are not causation. To make causal claims -- *what happens to output
when the central bank unexpectedly tightens policy?* -- we need to decompose the
correlated forecast errors into economically interpretable structural shocks.
This is the **identification problem**, and how you solve it determines what you
conclude.

This notebook demonstrates two fundamental identification strategies using the
canonical U.S. monetary policy application:

- **Cholesky decomposition** imposes a recursive causal ordering. It delivers a
  unique (point-identified) structural decomposition, but the results depend on
  which variable is ordered first.
- **Sign restrictions** impose qualitative constraints on the direction of
  responses. They deliver a *set* of admissible decompositions, producing wider
  credible bands that honestly reflect identification uncertainty.

By running both approaches on the same data and comparing the results, we make
five points concrete:

1. **Exact vs. set identification** -- Cholesky gives one answer; sign restrictions
   give a range of answers.
2. **Sensitivity to assumptions** -- Cholesky results change with variable ordering;
   sign restriction results change with the restriction horizon.
3. **Width of uncertainty** -- wider bands under sign restrictions are not a defect;
   they are honest.
4. **The price puzzle** -- a diagnostic signal that recursive identification may be
   failing.
5. **Economic content of restrictions** -- every identifying assumption tells an
   economic story. The reader must judge its plausibility.

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger("pytensor").setLevel(logging.ERROR)

In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from impulso import VAR, VARData, select_lag_order
from impulso.identification import Cholesky, SignRestriction
from impulso.samplers import NUTSSampler

## Data

The VAR system uses three variables from the canonical monetary policy SVAR literature
(Christiano, Eichenbaum, and Evans, 1999):

| Variable | Description | FRED Code | Transformation |
|----------|-------------|-----------|----------------|
| output | Industrial production index | `INDPRO` | Log level (x100) |
| prices | CPI, all urban consumers | `CPIAUCSL` | Log level (x100) |
| rate | Effective federal funds rate | `FEDFUNDS` | Level (%) |

The sample runs from **January 1965** to **December 2007**. The start date captures the
post-Accord period with active federal funds rate targeting. The end date excludes the
zero lower bound (ZLB) regime that began in December 2008, during which the federal
funds rate was constrained near zero and conventional monetary policy shock
identification breaks down.

Industrial production and CPI enter the VAR in log levels multiplied by 100, so that
coefficients can be interpreted in approximate percentage terms. The federal funds rate
enters in levels. We deliberately avoid differencing: Sims, Stock, and Watson (1990)
showed that Bayesian inference in levels VARs is valid regardless of integration
properties, and differencing can distort impulse responses if variables are cointegrated.

In [ ]:
from pathlib import Path

DATA_CACHE = Path("data/monetary_policy.csv")

if DATA_CACHE.exists():
    df = pd.read_csv(DATA_CACHE, index_col="date", parse_dates=True)
    print(f"Loaded cached data: {len(df)} observations")
else:
    from fredapi import Fred

    fred = Fred()  # Reads FRED_API_KEY from environment

    indpro = fred.get_series("INDPRO", observation_start="1965-01-01", observation_end="2007-12-31")
    cpi = fred.get_series("CPIAUCSL", observation_start="1965-01-01", observation_end="2007-12-31")
    fedfunds = fred.get_series("FEDFUNDS", observation_start="1965-01-01", observation_end="2007-12-31")

    df = pd.DataFrame({
        "output": np.log(indpro) * 100,
        "prices": np.log(cpi) * 100,
        "rate": fedfunds,
    }).dropna()
    df.index.name = "date"

    DATA_CACHE.parent.mkdir(exist_ok=True)
    df.to_csv(DATA_CACHE)
    print(f"Downloaded and cached {len(df)} observations")

df.describe().round(2)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
labels = {"output": "Log Industrial Production (x100)", "prices": "Log CPI (x100)", "rate": "Federal Funds Rate (%)"}

for ax, col in zip(axes, df.columns, strict=True):
    ax.plot(df.index, df[col], linewidth=0.6, color="0.3")
    ax.set_ylabel(labels[col], fontsize=9)
    ax.grid(alpha=0.3)

axes[0].set_title("U.S. Monetary Policy VAR -- Raw Data (1965-2007)")
fig.tight_layout()

## Reduced-Form VAR Estimation

The reduced-form VAR(p) model is:

$$y_t = c + A_1 y_{t-1} + A_2 y_{t-2} + \cdots + A_p y_{t-p} + u_t$$

where $y_t$ is the 3x1 vector of endogenous variables, $c$ is a vector of intercepts,
$A_1, \ldots, A_p$ are 3x3 coefficient matrices, and $u_t \sim N(0, \Sigma_u)$ is the
vector of reduced-form residuals. The residuals are correlated across equations because
they are linear combinations of the underlying structural shocks.

We use a **Minnesota (Litterman) prior**, which shrinks the VAR coefficients toward a
random walk. Own lags receive more prior weight than cross-variable lags, and higher
lags are shrunk more aggressively. This regularisation is particularly important for
monthly VARs with many lags, where the number of parameters can easily exceed the
number of observations.

In [ ]:
data = VARData.from_df(df, endog=["output", "prices", "rate"])

ic = select_lag_order(data, max_lags=14)
print(f"AIC: {ic.aic} lags, BIC: {ic.bic} lags, HQ: {ic.hq} lags")
ic.summary()

The literature standard for monthly monetary policy VARs is 12 lags, which allows the
model to capture up to one year of dynamic feedback. We use 12 lags for comparability
with published results (Christiano, Eichenbaum, and Evans, 1999; Uhlig, 2005),
noting the information criteria results above as a robustness check.

In [ ]:
sampler = NUTSSampler(
    draws=2000,
    tune=1000,
    chains=4,
    cores=1,
    random_seed=42,
    target_accept=0.9,
)
fitted = VAR(lags=12, prior="minnesota").fit(data, sampler=sampler)
fitted

In [ ]:
summary = az.summary(fitted.idata, var_names=["intercept"], kind="diagnostics")
print(f"Max R-hat: {summary['r_hat'].max():.3f}")
print(f"Min ESS (bulk): {summary['ess_bulk'].min():.0f}")
print(f"Divergences: {fitted.idata.sample_stats['diverging'].sum().values}")

## The Identification Problem

The reduced-form residuals $u_t$ are correlated across equations. The structural model
posits that they are linear combinations of orthogonal structural shocks $\varepsilon_t$:

$$u_t = B_0 \varepsilon_t, \quad \varepsilon_t \sim N(0, I)$$

This implies $\Sigma_u = B_0 B_0'$. For $n = 3$ variables, $\Sigma_u$ has
$n(n+1)/2 = 6$ unique elements, but $B_0$ has $n^2 = 9$ unknown entries. That leaves
3 free parameters -- we need 3 additional restrictions to pin down $B_0$.

Two strategies for supplying these restrictions:

- **Cholesky decomposition** imposes 3 zero restrictions by requiring $B_0$ to be lower
  triangular. This is equivalent to assuming a recursive causal ordering among the
  variables.
- **Sign restrictions** impose inequality constraints (e.g. "a contractionary monetary
  shock raises the interest rate and lowers prices") and accept *all* $B_0$ matrices
  consistent with these constraints.

The choice between these strategies is not statistical -- it is economic. Each approach
embeds assumptions about the structure of the economy, and those assumptions determine
the conclusions.

## Cholesky Identification

The baseline ordering is: **output, prices, rate** ($y_t$, $p_t$, $i_t$).

This embeds the following economic assumptions: real output and prices are "sluggish"
and cannot respond to a monetary policy shock within the same month. The central bank,
by contrast, observes current-month output and price conditions and adjusts the policy
rate accordingly. The monetary policy shock is the residual variation in the federal
funds rate equation after accounting for the systematic response to output and prices.

The Cholesky decomposition imposes $n(n-1)/2 = 3$ zero restrictions on the
contemporaneous impact matrix. These are strong restrictions with specific economic
content. "Just running a Cholesky" is not assumption-free -- it is one of the most
restrictive identification strategies available. Its appeal is computational simplicity
and point identification, not agnosticism.

In [ ]:
ordering_a = ["output", "prices", "rate"]
identified_chol_a = fitted.set_identification_strategy(Cholesky(ordering=ordering_a))

irf_chol_a = identified_chol_a.impulse_response(horizon=48)
fig = irf_chol_a.plot()
fig.suptitle("Cholesky IRFs -- Ordering A: Output, Prices, Rate", y=1.02)

Look at the CPI response to a contractionary monetary shock (the rate shock column,
prices row). If prices *rise* following the shock, this is the **price puzzle**: a
contractionary monetary policy shock that is supposed to reduce inflation instead
appears to increase it. This is one of the most discussed anomalies in the monetary
VAR literature.

The price puzzle arises because the Cholesky-identified "monetary policy shock" may be
contaminated by the Fed's systematic forward-looking response to anticipated inflation.
If the Fed raises rates because it expects inflation to rise, and the three-variable
VAR cannot capture that anticipation, then what looks like a contractionary shock is
partly an endogenous response to inflationary pressure -- and the subsequent price
increase is not puzzling at all.

In [ ]:
fevd_chol_a = identified_chol_a.fevd(horizon=48)
fig = fevd_chol_a.plot()
fig.suptitle("FEVD -- Cholesky Ordering A", y=1.02)

### Sensitivity to variable ordering

Cholesky identification is not assumption-free. The ordering is a substantive economic
claim, and the results are only as credible as that claim. We now re-estimate with two
alternative orderings to see how much the results change:

- **Ordering A (baseline):** output, prices, rate -- output and prices are sluggish
- **Ordering B:** prices, output, rate -- prices before output
- **Ordering C:** rate, output, prices -- rate most exogenous (economically implausible
  for a central bank with access to real-time data)

In [ ]:
# Ordering B: prices, output, rate
ordering_b = ["prices", "output", "rate"]
identified_chol_b = fitted.set_identification_strategy(Cholesky(ordering=ordering_b))
irf_chol_b = identified_chol_b.impulse_response(horizon=48)

# Ordering C: rate, output, prices (rate most exogenous)
ordering_c = ["rate", "output", "prices"]
identified_chol_c = fitted.set_identification_strategy(Cholesky(ordering=ordering_c))
irf_chol_c = identified_chol_c.impulse_response(horizon=48)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
response_vars = ["output", "prices", "rate"]
response_labels = {"output": "Industrial Production", "prices": "CPI", "rate": "Federal Funds Rate"}
horizons = np.arange(49)

for i, resp in enumerate(response_vars):
    ax = axes[i]

    for label, irf_result, colour in [
        ("A: y,p,i", irf_chol_a, "C0"),
        ("B: p,y,i", irf_chol_b, "C1"),
        ("C: i,y,p", irf_chol_c, "C3"),
    ]:
        irf_data = irf_result.idata.posterior_predictive["irf"]
        med_vals = irf_data.sel(response=resp, shock="rate").median(dim=("chain", "draw")).values
        hdi_vals = az.hdi(irf_data.sel(response=resp, shock="rate"), hdi_prob=0.68)

        ax.plot(horizons, med_vals, color=colour, label=label, linewidth=1.5)
        ax.fill_between(
            horizons,
            hdi_vals.sel(hdi="lower").values,
            hdi_vals.sel(hdi="higher").values,
            alpha=0.15,
            color=colour,
        )

    ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")
    ax.set_ylabel(response_labels[resp])
    if i == 0:
        ax.legend(fontsize=9, title="Ordering")
        ax.set_title("Response to Contractionary Monetary Policy Shock -- Cholesky Orderings")

axes[-1].set_xlabel("Months")
fig.tight_layout()

The comparison across orderings makes the key point: the IRFs change -- potentially
substantially -- depending on the ordering. Ordering C (rate first) is economically
implausible but illustrates the extreme: treating monetary policy as exogenous to
everything within the month produces very different dynamics. The difference between
orderings A and B is subtler but still economically meaningful. These are not
statistical differences -- they are consequences of different economic assumptions
baked into the ordering.

## Sign Restrictions

Sign restrictions identify structural shocks by imposing qualitative constraints on the
*direction* of impulse responses, rather than zero restrictions on the contemporaneous
impact matrix. Instead of asserting that certain variables do not respond at all to a
shock within the period, the researcher asserts only the direction of certain responses.

The algorithm (Rubio-Ramirez, Waggoner, and Zha, 2010): for each posterior draw, sample
orthogonal rotation matrices $Q$ from the uniform (Haar) measure on $O(n)$, form the
candidate structural impact matrix $B_0 = \text{chol}(\Sigma_u) \times Q$, compute the
implied impulse responses at all required horizons, and retain only those draws where
the sign conditions are satisfied.

Our baseline restriction set follows Uhlig (2005):

| Variable | Sign restriction | Economic rationale |
|----------|------------------|--------------------|
| rate | Positive ($\geq 0$) | A contractionary shock raises the policy rate |
| prices | Non-positive ($\leq 0$) | Contractionary policy should not increase prices |
| output | **Unrestricted** | The output effect is what we want to learn |

The **restriction horizon** $h$ controls how many periods the sign conditions must hold.
We start with $h = 6$ months (Uhlig's baseline), then explore sensitivity to $h = 0$
and $h = 12$.

In [ ]:
scheme_h6 = SignRestriction(
    restrictions={
        "rate": {"monetary_policy": "+"},
        "prices": {"monetary_policy": "-"},
    },
    n_rotations=2000,
    restriction_horizon=6,
    random_seed=42,
)

identified_sr_h6 = fitted.set_identification_strategy(scheme_h6)
acceptance_rate = identified_sr_h6.idata.posterior.attrs.get("sign_restriction_acceptance_rate", "N/A")
print(
    f"Acceptance rate: {acceptance_rate:.1%}"
    if isinstance(acceptance_rate, float)
    else f"Acceptance rate: {acceptance_rate}"
)

In [ ]:
irf_sr_h6 = identified_sr_h6.impulse_response(horizon=48)
fig = irf_sr_h6.plot()
fig.suptitle("Sign Restriction IRFs -- h=6 (Uhlig baseline)", y=1.02)

Is the output response clearly negative? Uhlig (2005) found that the output response
was ambiguous under these restrictions -- the credible set included both positive and
negative responses. If this is replicated here, it tells us that the sign restrictions
on prices and the interest rate alone may not be informative enough to pin down the
monetary transmission mechanism.

The wider credible bands compared to Cholesky reflect **identification uncertainty** on
top of parameter uncertainty. This additional width is not a defect -- it is an honest
representation of how much we do not know given only qualitative sign restrictions.

### Restriction horizon sensitivity

The restriction horizon $h$ is a choice variable. A longer horizon shrinks the
identified set (fewer structural models are admissible), which tightens the credible
bands but at the cost of stronger assumptions. This is the sign restriction analogue
of ordering sensitivity in Cholesky.

In [ ]:
results_sr = {}
for h in [0, 6, 12]:
    scheme = SignRestriction(
        restrictions={
            "rate": {"monetary_policy": "+"},
            "prices": {"monetary_policy": "-"},
        },
        n_rotations=2000,
        restriction_horizon=h,
        random_seed=42,
    )
    ident = fitted.set_identification_strategy(scheme)
    ar = ident.idata.posterior.attrs.get("sign_restriction_acceptance_rate", float("nan"))
    results_sr[h] = {
        "identified": ident,
        "irf": ident.impulse_response(horizon=48),
        "acceptance_rate": ar,
    }
    print(f"h={h:>2}: acceptance rate = {ar:.1%}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

for i, resp in enumerate(response_vars):
    ax = axes[i]

    for h, colour in [(0, "C0"), (6, "C1"), (12, "C3")]:
        irf_data = results_sr[h]["irf"].idata.posterior_predictive["irf"]
        shock_coords = list(irf_data.coords["shock"].values)
        # Use first shock column (the identified monetary policy shock)
        shock_name = shock_coords[0]
        med_vals = irf_data.sel(response=resp, shock=shock_name).median(dim=("chain", "draw")).values

        ax.plot(horizons, med_vals, color=colour, label=f"h={h}", linewidth=1.5)

    ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")
    ax.set_ylabel(response_labels[resp])
    if i == 0:
        ax.legend(fontsize=9, title="Restriction horizon")
        ax.set_title("Response to Contractionary MP Shock -- Sign Restrictions")

axes[-1].set_xlabel("Months")
fig.tight_layout()

As expected, longer restriction horizons shrink the identified set: the acceptance rate
falls and the median responses tighten. With $h = 0$ (impact only), the restrictions
are very permissive and the identified set is large. With $h = 12$, the restrictions
are demanding and far fewer rotations satisfy them. Every identification strategy
involves choices that affect the results -- this is the sign restriction version of
that lesson.

## Comparative Analysis

The pedagogical payoff: side-by-side comparison reveals what each approach assumes and
how those assumptions shape conclusions.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

# Cholesky baseline (ordering A)
irf_chol_data = irf_chol_a.idata.posterior_predictive["irf"]
# Sign restriction baseline (h=6)
irf_sr_data = results_sr[6]["irf"].idata.posterior_predictive["irf"]
sr_shock_name = next(iter(irf_sr_data.coords["shock"].values))

for i, resp in enumerate(response_vars):
    ax = axes[i]

    # Cholesky: shock = "rate" (monetary policy shock in ordering A)
    chol_med = irf_chol_data.sel(response=resp, shock="rate").median(dim=("chain", "draw")).values
    chol_hdi = az.hdi(irf_chol_data.sel(response=resp, shock="rate"), hdi_prob=0.68)
    ax.plot(horizons, chol_med, color="C0", linewidth=2, label="Cholesky")
    ax.fill_between(
        horizons, chol_hdi.sel(hdi="lower").values, chol_hdi.sel(hdi="higher").values, alpha=0.2, color="C0"
    )

    # Sign restrictions: first shock column (monetary policy)
    sr_med = irf_sr_data.sel(response=resp, shock=sr_shock_name).median(dim=("chain", "draw")).values
    sr_hdi = az.hdi(irf_sr_data.sel(response=resp, shock=sr_shock_name), hdi_prob=0.68)
    ax.plot(horizons, sr_med, color="C3", linewidth=2, linestyle="--", label="Sign Restrictions (h=6)")
    ax.fill_between(horizons, sr_hdi.sel(hdi="lower").values, sr_hdi.sel(hdi="higher").values, alpha=0.2, color="C3")

    ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")
    ax.set_ylabel(response_labels[resp])
    if i == 0:
        ax.legend(fontsize=9)
        ax.set_title("Monetary Policy Shock: Cholesky vs Sign Restrictions")

axes[-1].set_xlabel("Months")
fig.tight_layout()

**The price puzzle comparison.** Does the price puzzle appear under Cholesky? Under sign
restrictions? Sign restrictions with prices restricted to be non-positive *cannot*
produce a price puzzle by construction -- but that is because the restriction was
imposed, not because the data resolved the puzzle. This is a crucial methodological
point: sign restrictions do not *solve* the price puzzle; they *assume it away*. Whether
this is appropriate depends on one's economic priors.

**Band width.** The wider sign restriction bands reflect identification uncertainty on
top of parameter uncertainty. Narrow Cholesky bands may reflect false precision if the
identifying assumptions are wrong. Wide sign restriction bands may be too wide for
policy guidance. There is no free lunch.

In [ ]:
rows = []

# Cholesky orderings
for name, irf_result in [
    ("Cholesky A (y,p,i)", irf_chol_a),
    ("Cholesky B (p,y,i)", irf_chol_b),
    ("Cholesky C (i,y,p)", irf_chol_c),
]:
    irf_data = irf_result.idata.posterior_predictive["irf"]
    output_resp = irf_data.sel(response="output", shock="rate").median(dim=("chain", "draw")).values
    prices_resp = irf_data.sel(response="prices", shock="rate").median(dim=("chain", "draw")).values

    rows.append({
        "Specification": name,
        "Peak Output Response": f"{output_resp.min():.3f}",
        "Month of Peak": int(output_resp.argmin()),
        "Price Puzzle (months > 0)": int((prices_resp > 0).sum()),
    })

# Sign restriction horizons
for h in [0, 6, 12]:
    irf_data = results_sr[h]["irf"].idata.posterior_predictive["irf"]
    sr_shock = next(iter(irf_data.coords["shock"].values))
    output_resp = irf_data.sel(response="output", shock=sr_shock).median(dim=("chain", "draw")).values
    ar = results_sr[h]["acceptance_rate"]

    rows.append({
        "Specification": f"Sign Restr. h={h} (AR={ar:.0%})",
        "Peak Output Response": f"{output_resp.min():.3f}",
        "Month of Peak": int(output_resp.argmin()),
        "Price Puzzle (months > 0)": "N/A (restricted)",
    })

pd.DataFrame(rows)

## Discussion and Conclusions

### When to use Cholesky

When you have strong economic theory for a recursive ordering and want point-identified,
precise estimates. Best when the contemporaneous exclusion restrictions are credible --
for example, when institutional knowledge or data timing clearly establishes that some
variables cannot respond to others within the observation period.

### When to use sign restrictions

When zero restrictions are too strong but you have qualitative priors about directions
of responses. Best when you want honest uncertainty that reflects identification
ambiguity. Particularly useful when the economic question is about the *sign* of a
response (e.g. "does output fall after a monetary contraction?") rather than its
precise magnitude.

### Neither approach is neutral

Cholesky imposes zero restrictions with specific economic content. Sign restrictions
impose inequality restrictions with their own economic content, plus a uniform prior
over rotations. Baumeister and Hamilton (2015) showed that this uniform prior over $Q$
is itself informative -- it implicitly places a non-uniform prior over the structural
parameters.

### The price puzzle is diagnostic

Its appearance under Cholesky signals possible misspecification: the recursive
identification may be contaminated by the Fed's systematic response to anticipated
inflation. Its absence under sign restrictions is by construction, not resolution.

### Limitations

- The 3-variable system is minimal. Larger systems (adding commodity prices, money
  supply, or long rates) may resolve puzzles that arise from omitted variables.
- Monthly industrial production covers only part of the economy, excluding services.
- Results are sample-dependent.
- The uniform prior over $Q$ in sign restrictions is itself informative.

### Extensions

- **Larger VARs:** Add commodity prices to help resolve the price puzzle
  (Christiano, Eichenbaum, and Evans, 1999).
- **Narrative restrictions:** Combine sign restrictions with narrative evidence about
  specific historical episodes (Antolin-Diaz and Rubio-Ramirez, 2018).
- **Time-varying parameters:** Allow the monetary transmission mechanism to evolve.
- **Bayesian proxy SVARs:** Use external instruments to achieve identification.

## References

- Baumeister, C. and Hamilton, J. D. (2015). Sign restrictions, structural vector autoregressions, and useful prior information. *Econometrica*, 83(5), 1963-1999.
- Christiano, L. J., Eichenbaum, M., and Evans, C. L. (1999). Monetary policy shocks: What have we learned and to what end? In *Handbook of Macroeconomics*, Vol. 1A, 65-148.
- Doan, T., Litterman, R., and Sims, C. (1984). Forecasting and conditional projection using realistic prior distributions. *Econometric Reviews*, 3(1), 1-44.
- Rubio-Ramirez, J. F., Waggoner, D. F., and Zha, T. (2010). Structural vector autoregressions: Theory of identification and algorithms for inference. *Review of Economic Studies*, 77(2), 665-696.
- Sims, C. A. (1980). Macroeconomics and reality. *Econometrica*, 48(1), 1-48.
- Sims, C. A., Stock, J. H., and Watson, M. W. (1990). Inference in linear time series models with some unit roots. *Econometrica*, 58(1), 113-144.
- Uhlig, H. (2005). What are the effects of monetary policy on output? Results from an agnostic identification procedure. *Journal of Monetary Economics*, 52(2), 381-419.